# Statistical analysis of the composition geometry for normalised-sum injection

Normalised sum-injection fixes the total injected norm. Writing $s = \hat{v}_i + \hat{v}_j$ for the unnormalised sum of the two unit behaviour directions, we inject

$$\delta_{\text{norm}} = \alpha\,\frac{s}{\lVert s \rVert},$$

so that $\lVert \delta_{\text{norm}} \rVert = \alpha$ regardless of the angle $\theta_{ij}$ between the two directions. The per-behaviour component along each direction still varies with the angle.

In [18]:
import json
from pathlib import Path
import pandas as pd
import numpy as np

REPO_ROOT = Path.cwd().parents[1]

In [19]:
# Load data
with open(REPO_ROOT / "results/composition/v2_phase125_normTrue_a4.5/scoring/summary.json") as f:
    summary = json.load(f)
  
# Metadata  
print(summary["model"], summary["layer"], summary["alpha"], summary["tau_value"])

# Per-composition entries
pairs = summary["pairs"]
print(pairs[0].keys)

# Extract per-composition data
rows = []
for p in pairs:
    if p.get("status") != "ok":
        continue
    rows.append({
        "trait_a": p["trait_a"], "trait_b": p["trait_b"],
        "cos": p["cos"], "regime": p["regime"],
        "comp_base": p["baseline"]["composition_mean"],
        "comp_steered": p["steered"]["composition_mean"],
        "coh_steered": p["steered"]["coherence_mean"],
        "delta_a_joint":  p["delta"]["trait_a_joint"],
        "delta_a_single": p["delta"]["trait_a_single"],
        "delta_b_joint":  p["delta"]["trait_b_joint"],
        "delta_b_single": p["delta"]["trait_b_single"],
        "delta_comp": p["delta"]["composition"],
        "delta_coh":  p["delta"]["coherence"],
    })
    
# Convert to pd dataframe
df = pd.DataFrame(rows)
df["regime"].value_counts()

meta-llama/Llama-3.1-8B-Instruct 17 4.5 0.8296650044918062
<built-in method keys of dict object at 0x12ced74c0>


regime
mixed          14
additive        7
suppressive     4
dominant        3
Name: count, dtype: int64

A natural first analysis bins each pair into a discrete composition regime (roughly, whether the joint injection expressed both behaviours, one, or neither) and asks whether the absolute angle $|cos θ_{ij}|$ predicts the regime.

In [20]:
# Look at dataframe
df.head(n=40)

,trait_a,trait_b,cos,regime,comp_base,comp_steered,coh_steered,delta_a_joint,delta_a_single,delta_b_joint,delta_b_single,delta_comp,delta_coh
0,apathetic,confidence,0.0143,dominant,33.72,52.65,79.80,33.25,41.36,4.61,21.87,18.93,-18.37
1,apathetic,evil,0.2578,dominant,3.97,40.03,50.35,58.34,45.50,13.79,63.48,36.06,-46.93
2,apathetic,formality,0.0150,mixed,45.47,69.47,89.00,44.98,30.64,3.00,6.18,23.99,-8.84
3,apathetic,hallucinating,-0.1642,additive,7.77,43.88,69.54,28.98,38.16,43.24,47.25,36.11,-23.55
4,apathetic,humorous,0.1923,mixed,1.35,53.79,49.15,60.36,30.84,44.51,78.57,52.44,-48.05
5,apathetic,impolite,0.6948,mixed,1.09,54.43,55.92,55.51,35.12,51.16,72.44,53.33,-40.70
6,apathetic,sycophantic,0.0503,mixed,4.39,34.49,65.50,35.79,42.21,24.42,79.85,30.10,-32.28
7,confidence,evil,0.2294,mixed,28.32,48.14,63.91,9.52,25.28,30.13,53.33,19.82,-32.45
8,confidence,formality,0.2262,mixed,77.59,85.44,92.13,12.28,18.80,3.42,4.29,7.85,-5.50
9,confidence,hallucinating,0.2700,additive,26.35,56.93,81.24,21.42,23.73,38.87,37.53,30.59,-11.37


### Cross-tab for cosine v. regime

We perform a descriptive analysis to see whether simple cross-tabulation predicts a regime using cosine similarity (Gram) matrix $G$.

In [21]:
# Add column with absolute value of cosine similarities
df['cosine_abs'] = df['cos'].abs()

# Bin cosine with the same strata of the previous analysis
df['cos_bin'] = pd.cut(
    df['cosine_abs'],
    bins=[0, 0.2, 0.5, 1.0],
    labels=['low', 'moderate', 'high']
)

df.head(n=40)

,trait_a,trait_b,cos,regime,comp_base,comp_steered,coh_steered,delta_a_joint,delta_a_single,delta_b_joint,delta_b_single,delta_comp,delta_coh,cosine_abs,cos_bin
0,apathetic,confidence,0.0143,dominant,33.72,52.65,79.80,33.25,41.36,4.61,21.87,18.93,-18.37,0.0143,low
1,apathetic,evil,0.2578,dominant,3.97,40.03,50.35,58.34,45.50,13.79,63.48,36.06,-46.93,0.2578,moderate
2,apathetic,formality,0.0150,mixed,45.47,69.47,89.00,44.98,30.64,3.00,6.18,23.99,-8.84,0.0150,low
3,apathetic,hallucinating,-0.1642,additive,7.77,43.88,69.54,28.98,38.16,43.24,47.25,36.11,-23.55,0.1642,low
4,apathetic,humorous,0.1923,mixed,1.35,53.79,49.15,60.36,30.84,44.51,78.57,52.44,-48.05,0.1923,low
5,apathetic,impolite,0.6948,mixed,1.09,54.43,55.92,55.51,35.12,51.16,72.44,53.33,-40.70,0.6948,high
6,apathetic,sycophantic,0.0503,mixed,4.39,34.49,65.50,35.79,42.21,24.42,79.85,30.10,-32.28,0.0503,low
7,confidence,evil,0.2294,mixed,28.32,48.14,63.91,9.52,25.28,30.13,53.33,19.82,-32.45,0.2294,moderate
8,confidence,formality,0.2262,mixed,77.59,85.44,92.13,12.28,18.80,3.42,4.29,7.85,-5.50,0.2262,moderate
9,confidence,hallucinating,0.2700,additive,26.35,56.93,81.24,21.42,23.73,38.87,37.53,30.59,-11.37,0.2700,moderate


In [22]:
# Cross-tab
pd.crosstab(df['cos_bin'], df['regime'])

regime,additive,dominant,mixed,suppressive
cos_bin,,,,
low,2,1,7,2
moderate,5,2,6,1
high,0,0,1,1


In [ ]:
# Small-sample association test on the cos_bin x regime cross-tab.
# At n=28 with ~10/12 cells having expected count < 5, the chi-square approximation
# is invalid. scipy's fisher_exact supports only 2x2 tables, so we use two forms:
#   (1) full r x c table  -> Monte-Carlo "exact" test (no expected-count assumption)
#   (2) interpretable 2x2 -> a genuine Fisher's exact test
from scipy.stats import fisher_exact, chi2_contingency

ct = pd.crosstab(df['cos_bin'], df['regime'])
print("Cross-tab (cos_bin x regime):")
print(ct, "\n")

# --- (1) Full table: Monte-Carlo exact test ---------------------------------
# Same hypothesis as the chi-square ("does the cosine bin associate with regime?"),
# but the null is built by permuting regime labels rather than assuming the
# asymptotic chi-square distribution. Valid for sparse tables.
sub      = df.dropna(subset=['cos_bin'])
bin_code = pd.Categorical(sub['cos_bin']).codes
reg_code = pd.Categorical(sub['regime']).codes
n_bin, n_reg = bin_code.max() + 1, reg_code.max() + 1

from bins_utils import _chi2_stat

rng_mc   = np.random.default_rng(0)
N_MC     = 10000
obs_chi2 = _chi2_stat(bin_code, reg_code, n_bin, n_reg)
perm     = np.fromiter((_chi2_stat(bin_code, rng_mc.permutation(reg_code))
                        for _ in range(N_MC)), float, N_MC)
p_mc     = (np.sum(perm >= obs_chi2) + 1) / (N_MC + 1)
print(f"(1) Full {ct.shape[0]}x{ct.shape[1]} table - Monte-Carlo exact test "
      f"({N_MC} permutations):")
print(f"    observed chi2 = {obs_chi2:.3f},  p = {p_mc:.4f}\n")

# --- (2) Interpretable 2x2 Fisher's exact -----------------------------------
# Collapse to the binary question the rest of the notebook cares about:
# near-orthogonal vs aligned  x  additive vs not.
df['cos_hi']      = (df['cosine_abs'] >= 0.2).astype(int)   # 0 = |cos|<0.2, 1 = >=0.2
additive          = (df['regime'] == 'additive').astype(int)
tab2              = pd.crosstab(df['cos_hi'], additive)
tab2.index.name   = '|cos|>=0.2'
tab2.columns.name = 'is_additive'
odds, p_fisher    = fisher_exact(tab2, alternative='two-sided')
print("(2) 2x2 Fisher's exact (|cos|>=0.2  x  is_additive):")
print(tab2)
print(f"    odds ratio = {odds:.3f},  p = {p_fisher:.4f}")


Cross-tab (cos_bin x regime):
regime    additive  dominant  mixed  suppressive
cos_bin                                         
low              2         1      7            2
moderate         5         2      6            1
high             0         0      1            1 

(1) Full 3x4 table - Monte-Carlo exact test (10000 permutations):
    observed chi2 = 4.563,  p = 0.6672

(2) 2x2 Fisher's exact (|cos|>=0.2  x  is_additive):
is_additive   0  1
|cos|>=0.2        
0            10  2
1            11  5
    odds ratio = 2.273,  p = 0.6618


The cross-tab between cosine bin and regime shows no significant geometry–regime association under either an exact 3×4 test (p = 0.67) or a collapsed 2×2 test (p = 0.66). With the majority of vectors being nearly orthogonal (i.e. in the low/moderate cosine bins), the high-similarity stratum is nearly empty. 

### Correlation between cosine and semantic similarities
We compute Pearson correlation between absolute cosine similarity and semantic similarity, to understand if the latter could be a confound.

In [24]:
# Add semantic similarity
with open(REPO_ROOT / "results/semantic_similarity.json") as f:
    sem = json.load(f)
    
sem_lookup = {
    tuple(sorted([p["trait_a"], p["trait_b"]])): p["sem_sim"]
    for p in sem["pairs"]
}

df["sem_sim"] = df.apply(
    lambda r: sem_lookup[tuple(sorted([r["trait_a"], r["trait_b"]]))],
    axis=1,
)

In [25]:
from scipy.stats import pearsonr

# Confound check (single source of truth): is semantic similarity collinear with |cos|?
# r_cs / p_cs are reused by the partial-Spearman motivation and the RQ1 summary save.
r_cs, p_cs = pearsonr(df["cosine_abs"], df["sem_sim"])
print(f"Pearson r(cosine_abs, sem_sim) = {r_cs:+.3f}  (p = {p_cs:.4f})")
print(df[["cosine_abs", "sem_sim"]].describe().round(3))

Pearson r(cosine_abs, sem_sim) = +0.532  (p = 0.0036)
       cosine_abs  sem_sim
count      28.000   28.000
mean        0.242    0.276
std         0.162    0.077
min         0.014    0.185
25%         0.120    0.222
50%         0.230    0.245
75%         0.358    0.322
max         0.695    0.470


With a correlation over +0.5, we should account for similarity as a confound for our analysis when using absolute cosine similarity.
On the other hand, semantic similarity should not be a confounder for signed cosine similarity, since the former is always positive.

In [26]:
corr = df[["cos", "sem_sim"]].corr().iloc[0, 1]
print(f"Pearson r(cos, sem_sim) = {corr:+.3f}")
print(df[["cos", "sem_sim"]].describe().round(3))

Pearson r(cos, sem_sim) = +0.135
          cos  sem_sim
count  28.000   28.000
mean    0.151    0.276
std     0.252    0.077
min    -0.522    0.185
25%    -0.002    0.222
50%     0.209    0.245
75%     0.305    0.322
max     0.695    0.470


In [27]:
# Save experiments results
results = []

In [ ]:
from sklearn.base import clone
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score
from sklearn.model_selection import cross_val_predict, StratifiedKFold, LeaveOneOut

from bins_utils import bootstrap_perm, bootstrap_perm_multi

# Single rng instantiated
rng = np.random.default_rng(0)
N_BOOT  = 2000
N_PERM  = 10000

In [ ]:
scaler = StandardScaler()
multi_model = LogisticRegression(penalty=None)

## Multinomial logistic regression

We keep the discrete 4-class `regime` (additive / dominant / mixed / suppressive) as the target (the same bins the cross-tab above used) and ask whether geometry can predict *which* regime a pair falls into. AUC is macro one-vs-rest. With n = 28 across 4 classes this is heavily underpowered, so it is exploratory; the point is to show the discrete-regime target is not recoverable from geometry, which motivates collapsing to the binary additive-vs-non-additive question in the next section.

Experiments 1–2 use the **signed** cosine (alignment sign retained); experiments 3–4 use the **absolute** cosine. In each case we also report a variant with semantic similarity added as a confound control.

| # | Outcome | Predictors |
|---|---------|------------|
| 1 | regime | signed $\cos$ |
| 2 | regime | signed $\cos$ + sem\_sim |
| 3 | regime | $\lvert\cos\rvert$ |
| 4 | regime | $\lvert\cos\rvert$ + sem\_sim |

### Exp 1: Multinomial LR: signed $\cos$ → regime

The signed cosine keeps the alignment sign (aligned $\to +$, opposed $\to -$). Tested against the full 4-class regime, it asks whether the *direction* of alignment, not just its magnitude, separates the regimes. AUC is macro one-vs-rest.

In [31]:
y_multi      = df["regime"].to_numpy()
X_sgn_s_m    = scaler.fit_transform(df[['cos']].to_numpy())

m1_results   = multi_model.fit(X_sgn_s_m, y_multi)
probas_in_m1 = m1_results.predict_proba(X_sgn_s_m)
m1_auc       = roc_auc_score(y_multi, probas_in_m1, multi_class='ovr', average='macro')

loo            = LeaveOneOut()
y_proba_loo_m1 = cross_val_predict(multi_model, X_sgn_s_m, y_multi, cv=loo, method="predict_proba")
m1_auc_loo     = roc_auc_score(y_multi, y_proba_loo_m1, multi_class='ovr', average='macro')

print(f"in-sample AUC (macro OVR) = {m1_auc:.3f}")
print(f"LOO (macro OVR)           = {m1_auc_loo:.3f}")
for i, c in enumerate(m1_results.classes_):
    print(f"  class {c}: coef = {m1_results.coef_[i, 0]:+.3f}")

in-sample AUC (macro OVR) = 0.648
LOO (macro OVR)           = 0.327
  class additive: coef = +0.526
  class dominant: coef = +0.545
  class mixed: coef = +0.472
  class suppressive: coef = -1.543


/Users/federicoscaffidimuta/Desktop/Third year/ML project/steering-vector-composition/venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/Users/federicoscaffidimuta/Desktop/Third year/ML project/steering-vector-composition/venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/Users/federicoscaffidimuta/Desktop/Third year/ML project/steering-vector-composition/venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will a

In [ ]:
stats = bootstrap_perm_multi(multi_model, X_sgn_s_m, y_multi, m1_auc, N_BOOT, N_PERM, rng)
results.append({
    "experiment":  "M1_multi_signed_cos",
    "outcome":     "regime",
    "predictors":  "signed_cos",
    "n_features":  1,
    "auc_loo":     m1_auc_loo,
    **stats,
})

### Exp 2: Multinomial LR: signed $\cos$ + sem\_sim → regime

Adds the semantic baseline alongside signed cosine. Checks whether the two together separate the regimes better than signed geometry alone (Exp 1).

In [33]:
X_sgn_sem_m   = df[['cos', 'sem_sim']].to_numpy()
X_sgn_sem_s_m = scaler.fit_transform(X_sgn_sem_m)

m2_results    = multi_model.fit(X_sgn_sem_s_m, y_multi)
probas_in_m2  = m2_results.predict_proba(X_sgn_sem_s_m)
m2_auc        = roc_auc_score(y_multi, probas_in_m2, multi_class='ovr', average='macro')

loo            = LeaveOneOut()
y_proba_loo_m2 = cross_val_predict(multi_model, X_sgn_sem_s_m, y_multi, cv=loo, method="predict_proba")
m2_auc_loo     = roc_auc_score(y_multi, y_proba_loo_m2, multi_class='ovr', average='macro')

print(f"in-sample AUC (macro OVR) = {m2_auc:.3f}")
print(f"LOO (macro OVR)           = {m2_auc_loo:.3f}")
for i, c in enumerate(m2_results.classes_):
    print(f"  class {c}: coef_signed={m2_results.coef_[i,0]:+.3f}  coef_sem={m2_results.coef_[i,1]:+.3f}")

in-sample AUC (macro OVR) = 0.723
LOO (macro OVR)           = 0.409
  class additive: coef_signed=+0.801  coef_sem=-0.737
  class dominant: coef_signed=+0.566  coef_sem=-0.072
  class mixed: coef_signed=+0.274  coef_sem=+0.308
  class suppressive: coef_signed=-1.641  coef_sem=+0.501


/Users/federicoscaffidimuta/Desktop/Third year/ML project/steering-vector-composition/venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/Users/federicoscaffidimuta/Desktop/Third year/ML project/steering-vector-composition/venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/Users/federicoscaffidimuta/Desktop/Third year/ML project/steering-vector-composition/venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will a

In [ ]:
stats = bootstrap_perm_multi(multi_model, X_sgn_sem_s_m, y_multi, m2_auc, N_BOOT, N_PERM, rng)
results.append({
    "experiment":  "M2_multi_signed_sem",
    "outcome":     "regime",
    "predictors":  "signed_cos + sem_sim",
    "n_features":  2,
    "auc_loo":     m2_auc_loo,
    **stats,
})

### Exp 3: Multinomial LR: $|\cos|$ → regime

The absolute-cosine counterpart of Exp 1: does the *magnitude* of geometric overlap (sign discarded) separate the four regimes? AUC is macro one-vs-rest.

In [35]:
y_multi   = df["regime"].to_numpy()
X_abs_s_4 = scaler.fit_transform(df[['cosine_abs']].to_numpy())

exp4_results = multi_model.fit(X_abs_s_4, y_multi)
probas_in_4  = exp4_results.predict_proba(X_abs_s_4)
exp4_auc     = roc_auc_score(y_multi, probas_in_4, multi_class='ovr', average='macro')

loo           = LeaveOneOut()
y_proba_loo_4 = cross_val_predict(multi_model, X_abs_s_4, y_multi, cv=loo, method="predict_proba")
exp4_auc_loo  = roc_auc_score(y_multi, y_proba_loo_4, multi_class='ovr', average='macro')

print(f"in-sample AUC (macro OVR) = {exp4_auc:.3f}")
print(f"LOO (macro OVR)           = {exp4_auc_loo:.3f}")
for i, c in enumerate(exp4_results.classes_):
    print(f"  class {c}: coef = {exp4_results.coef_[i, 0]:+.3f}")

/Users/federicoscaffidimuta/Desktop/Third year/ML project/steering-vector-composition/venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/Users/federicoscaffidimuta/Desktop/Third year/ML project/steering-vector-composition/venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/Users/federicoscaffidimuta/Desktop/Third year/ML project/steering-vector-composition/venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will a

in-sample AUC (macro OVR) = 0.563
LOO (macro OVR)           = 0.079
  class additive: coef = +0.182
  class dominant: coef = -0.190
  class mixed: coef = -0.071
  class suppressive: coef = +0.079


/Users/federicoscaffidimuta/Desktop/Third year/ML project/steering-vector-composition/venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/Users/federicoscaffidimuta/Desktop/Third year/ML project/steering-vector-composition/venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/Users/federicoscaffidimuta/Desktop/Third year/ML project/steering-vector-composition/venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will a

In [ ]:
stats = bootstrap_perm_multi(multi_model, X_abs_s_4, y_multi, exp4_auc, N_BOOT, N_PERM, rng)
results.append({
    "experiment":  "M3_multi_abs_cos",
    "outcome":     "regime",
    "predictors":  "cosine_abs",
    "n_features":  1,
    "auc_loo":     exp4_auc_loo,
    **stats,
})

### Exp 4: Multinomial LR: $|\cos|$ + sem\_sim → regime

Full multinomial model with absolute cosine and semantics. Exploratory: checks whether the two signals together can discriminate among regimes better than absolute geometry alone (Exp 3).

In [37]:
X_both_7   = df[['cosine_abs', 'sem_sim']].to_numpy()
X_both_s_7 = scaler.fit_transform(X_both_7)

exp7_results = multi_model.fit(X_both_s_7, y_multi)
probas_in_7  = exp7_results.predict_proba(X_both_s_7)
exp7_auc     = roc_auc_score(y_multi, probas_in_7, multi_class='ovr', average='macro')

loo           = LeaveOneOut()
y_proba_loo_7 = cross_val_predict(multi_model, X_both_s_7, y_multi, cv=loo, method="predict_proba")
exp7_auc_loo  = roc_auc_score(y_multi, y_proba_loo_7, multi_class='ovr', average='macro')

print(f"in-sample AUC (macro OVR) = {exp7_auc:.3f}")
print(f"LOO (macro OVR)           = {exp7_auc_loo:.3f}")
for i, c in enumerate(exp7_results.classes_):
    print(f"  class {c}: coef_abs={exp7_results.coef_[i,0]:+.3f}  coef_sem={exp7_results.coef_[i,1]:+.3f}")

/Users/federicoscaffidimuta/Desktop/Third year/ML project/steering-vector-composition/venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/Users/federicoscaffidimuta/Desktop/Third year/ML project/steering-vector-composition/venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/Users/federicoscaffidimuta/Desktop/Third year/ML project/steering-vector-composition/venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will a

in-sample AUC (macro OVR) = 0.656
LOO (macro OVR)           = 0.296
  class additive: coef_abs=+0.681  coef_sem=-0.940
  class dominant: coef_abs=-0.199  coef_sem=+0.068
  class mixed: coef_abs=-0.274  coef_sem=+0.385
  class suppressive: coef_abs=-0.208  coef_sem=+0.487


/Users/federicoscaffidimuta/Desktop/Third year/ML project/steering-vector-composition/venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/Users/federicoscaffidimuta/Desktop/Third year/ML project/steering-vector-composition/venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/Users/federicoscaffidimuta/Desktop/Third year/ML project/steering-vector-composition/venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will a

In [ ]:
stats = bootstrap_perm_multi(multi_model, X_both_s_7, y_multi, exp7_auc, N_BOOT, N_PERM, rng)
results.append({
    "experiment":  "M4_multi_abs_sem",
    "outcome":     "regime",
    "predictors":  "cosine_abs + sem_sim",
    "n_features":  2,
    "auc_loo":     exp7_auc_loo,
    **stats,
})

## Binary logistic regression

Having seen that the discrete 4-class regime is not recoverable from geometry (above), we collapse the outcome to the binary **additive vs non-additive** question and run **four binary logistic regressions** (predicting `is_additive`) — the **same four predictor sets used in the multinomial section above**, now against the binary outcome.

Experiments 1–2 use the **absolute** cosine $|\cos|$ (magnitude of geometric overlap); Experiments 3–4 use the **signed** cosine (alignment sign retained). In each pair, the second model adds semantic similarity as a confound control: if the geometric slope survives — and LOO AUC does not collapse — when `sem_sim` is included, the geometric signal is real and not merely a semantic proxy.

| # | Outcome | Predictors |
|---|---------|------------|
| 1 | is\_additive | $\lvert\cos\rvert$ |
| 2 | is\_additive | $\lvert\cos\rvert$ + sem\_sim |
| 3 | is\_additive | signed $\cos$ |
| 4 | is\_additive | signed $\cos$ + sem\_sim |

#### Metrics

For each model we report four quantities:

| Metric | How it is computed | What it tells you |
|--------|--------------------|-------------------|
| **In-sample AUC** | ROC-AUC on the same 28 pairs used to fit the model | Optimistic upper bound — confirms the model can fit the data; not a measure of generalisation |
| **LOO AUC** | Pool the 28 held-out probabilities from leave-one-out CV, then compute one AUC | **Primary metric.** The honest out-of-sample estimate of predictive signal |
| **Bootstrap 95% CI** | 2 000 stratified resamples, each with a fresh model refit; percentile interval on in-sample AUC | Uncertainty around the AUC given the small sample size (n = 28) |
| **Permutation p** | Fraction of 10 000 label-shuffled AUCs ≥ observed, one-sided | Whether the predictor explains more variance than chance |

**Why AUC and not accuracy?** With 7 additive pairs out of 28, a classifier that always predicts "non-additive" reaches 75 % accuracy but AUC = 0.5. AUC is threshold-free and handles class imbalance.

**What counts as evidence?** LOO AUC > 0.7 *and* permutation p < 0.05 together. LOO AUC is the primary metric; in-sample AUC is shown for reference only.

In [39]:
# Add binary variable for additive v. non-additive
df['is_additive'] = (df['regime'] == 'additive').astype(int)
df.head()

,trait_a,trait_b,cos,regime,comp_base,comp_steered,coh_steered,delta_a_joint,delta_a_single,delta_b_joint,delta_b_single,delta_comp,delta_coh,cosine_abs,cos_bin,cos_hi,sem_sim,is_additive
0,apathetic,confidence,0.0143,dominant,33.72,52.65,79.80,33.25,41.36,4.61,21.87,18.93,-18.37,0.0143,low,0,0.240104,0
1,apathetic,evil,0.2578,dominant,3.97,40.03,50.35,58.34,45.50,13.79,63.48,36.06,-46.93,0.2578,moderate,1,0.321475,0
2,apathetic,formality,0.0150,mixed,45.47,69.47,89.00,44.98,30.64,3.00,6.18,23.99,-8.84,0.0150,low,0,0.246348,0
3,apathetic,hallucinating,-0.1642,additive,7.77,43.88,69.54,28.98,38.16,43.24,47.25,36.11,-23.55,0.1642,low,0,0.198113,1
4,apathetic,humorous,0.1923,mixed,1.35,53.79,49.15,60.36,30.84,44.51,78.57,52.44,-48.05,0.1923,low,0,0.232589,0


In [ ]:
binary_model = LogisticRegression(penalty=None)

### Exp 1: Binary LR: $|\cos|$ → is\_additive

Baseline geometric model. Tests whether pairs with more aligned steering vectors are *less* likely to compose additively, as the Park LRH predicts.

In [41]:
# Load exp 1 data
X_abs   = df[['cosine_abs']].to_numpy()
X_abs_s = scaler.fit_transform(X_abs)
y_add   = df["is_additive"].to_numpy()

# Fit on all data for in-sample AUC + coefficient sign
exp1_results = binary_model.fit(X_abs_s, y_add)
y_proba_in   = exp1_results.predict_proba(X_abs_s)[:, 1]
exp1_auc     = roc_auc_score(y_add, y_proba_in)

# Leave-one-out: fit on n-1, predict held-out probability, pool 36 probabilities, single AUC
loo            = LeaveOneOut()
y_proba_loo    = cross_val_predict(binary_model, X_abs_s, y_add, cv=loo, method="predict_proba")[:, 1]
exp1_auc_loo   = roc_auc_score(y_add, y_proba_loo)

print(f"in-sample AUC      = {exp1_auc:.3f}")
print(f"LOO (pooled) AUC   = {exp1_auc_loo:.3f}")
print(f"slope on cosine_abs (scaled) = {exp1_results.coef_[0,0]:+.3f}  "
      f"(negative => high |cos| reduces P(additive) — the geometric prediction)")

in-sample AUC      = 0.619
LOO (pooled) AUC   = 0.245
slope on cosine_abs (scaled) = +0.239  (negative => high |cos| reduces P(additive) — the geometric prediction)


In [ ]:
# Run CI bootstrap and p-value permutation
stats = bootstrap_perm(binary_model, X_abs_s, y_add, exp1_auc, N_BOOT, N_PERM, rng)
results.append({
    "experiment":  "B1_abs_cos",
    "outcome":     "is_additive",
    "predictors":  "cosine_abs",
    "n_features":  1,
    "auc_loo":     exp1_auc_loo,
    "slope_abs_cos": float(exp1_results.coef_[0, 0]),
    "slope_both_antisocial": None,         # not in this model
    "slope_signed_cos":      None,
    **stats,
})

LOO folds with < 2 positives in training: 0
  feature 0: slope min=+0.074  max=+0.541  n_negative=0/28
AUC = 0.619  95% CI [0.456, 0.830]  (n_boot=2000  shape: min=0.408 median=0.633 max=0.939)
permutation p (one-sided) = 0.3702


### Exp 2: Binary LR: $|\cos|$ + sem\_sim → is\_additive

Confound control for the absolute-cosine model. Both predictors in the same model. If $|\cos|$ retains a negative slope and LOO AUC does not drop relative to Exp 1 ($|\cos|$ alone), the geometric signal is real and not merely tracking semantic similarity.

In [43]:
X_both_5   = df[['cosine_abs', 'sem_sim']].to_numpy()
X_both_s_5 = scaler.fit_transform(X_both_5)

exp5_results = binary_model.fit(X_both_s_5, y_add)
y_proba_in_5 = exp5_results.predict_proba(X_both_s_5)[:, 1]
exp5_auc     = roc_auc_score(y_add, y_proba_in_5)

loo           = LeaveOneOut()
y_proba_loo_5 = cross_val_predict(binary_model, X_both_s_5, y_add, cv=loo, method="predict_proba")[:, 1]
exp5_auc_loo  = roc_auc_score(y_add, y_proba_loo_5)

print(f"in-sample AUC      = {exp5_auc:.3f}")
print(f"LOO (pooled) AUC   = {exp5_auc_loo:.3f}")
print(f"slope on cosine_abs (scaled) = {exp5_results.coef_[0,0]:+.3f}")
print(f"slope on sem_sim   (scaled)  = {exp5_results.coef_[0,1]:+.3f}")

in-sample AUC      = 0.741
LOO (pooled) AUC   = 0.653
slope on cosine_abs (scaled) = +0.930
slope on sem_sim   (scaled)  = -1.301


In [ ]:
stats = bootstrap_perm(binary_model, X_both_s_5, y_add, exp5_auc, N_BOOT, N_PERM, rng)
results.append({
    "experiment":  "B2_abs_cos_sem",
    "outcome":     "is_additive",
    "predictors":  "cosine_abs + sem_sim",
    "n_features":  2,
    "auc_loo":     exp5_auc_loo,
    "slope_abs_cos":         float(exp5_results.coef_[0, 0]),
    "slope_both_antisocial": None,
    "slope_signed_cos":      None,
    "slope_sem_sim":         float(exp5_results.coef_[0, 1]),
    **stats,
})

### Exp 3: Binary LR: signed $\cos$ → is\_additive

Sign sensitivity check. If additivity depends only on the magnitude of interference, signed cos should not outperform $|\cos|$ (Exp 1). A higher LOO AUC here would suggest direction matters too.

In [45]:
X_sgn   = df[['cos']].to_numpy()
X_sgn_s = scaler.fit_transform(X_sgn)
y_add   = df["is_additive"].to_numpy()

exp3_results = binary_model.fit(X_sgn_s, y_add)
y_proba_in_3 = exp3_results.predict_proba(X_sgn_s)[:, 1]
exp3_auc     = roc_auc_score(y_add, y_proba_in_3)

loo           = LeaveOneOut()
y_proba_loo_3 = cross_val_predict(binary_model, X_sgn_s, y_add, cv=loo, method="predict_proba")[:, 1]
exp3_auc_loo  = roc_auc_score(y_add, y_proba_loo_3)

print(f"in-sample AUC      = {exp3_auc:.3f}")
print(f"LOO (pooled) AUC   = {exp3_auc_loo:.3f}")
print(f"slope on signed_cos (scaled) = {exp3_results.coef_[0,0]:+.3f}  "
      f"(negative => high signed_cos reduces P(additive))")

in-sample AUC      = 0.619
LOO (pooled) AUC   = 0.333
slope on signed_cos (scaled) = +0.339  (negative => high signed_cos reduces P(additive))


In [ ]:
stats = bootstrap_perm(binary_model, X_sgn_s, y_add, exp3_auc, N_BOOT, N_PERM, rng)
results.append({
    "experiment":  "B3_signed_cos",
    "outcome":     "is_additive",
    "predictors":  "signed_cos",
    "n_features":  1,
    "auc_loo":     exp3_auc_loo,
    "slope_abs_cos":         None,
    "slope_both_antisocial": None,
    "slope_signed_cos":      float(exp3_results.coef_[0, 0]),
    **stats,
})

### Exp 4: Binary LR: signed $\cos$ + sem\_sim → is\_additive

Confound control for the signed-cosine model, mirroring Exp 2. Adds semantic similarity alongside signed cosine: checks whether the signed-geometry slope survives controlling for semantics, and whether the two together separate additive pairs better than signed cosine alone (Exp 3).

In [47]:
X_sgn_sem   = df[['cos', 'sem_sim']].to_numpy()
X_sgn_sem_s = scaler.fit_transform(X_sgn_sem)
y_add       = df["is_additive"].to_numpy()

exp8_results = binary_model.fit(X_sgn_sem_s, y_add)
y_proba_in_8 = exp8_results.predict_proba(X_sgn_sem_s)[:, 1]
exp8_auc     = roc_auc_score(y_add, y_proba_in_8)

loo           = LeaveOneOut()
y_proba_loo_8 = cross_val_predict(binary_model, X_sgn_sem_s, y_add, cv=loo, method="predict_proba")[:, 1]
exp8_auc_loo  = roc_auc_score(y_add, y_proba_loo_8)

print(f"in-sample AUC      = {exp8_auc:.3f}")
print(f"LOO (pooled) AUC   = {exp8_auc_loo:.3f}")
print(f"slope on signed_cos (scaled) = {exp8_results.coef_[0,0]:+.3f}")
print(f"slope on sem_sim   (scaled)  = {exp8_results.coef_[0,1]:+.3f}")

in-sample AUC      = 0.728
LOO (pooled) AUC   = 0.517
slope on signed_cos (scaled) = +0.691
slope on sem_sim   (scaled)  = -1.023


In [ ]:
stats = bootstrap_perm(binary_model, X_sgn_sem_s, y_add, exp8_auc, N_BOOT, N_PERM, rng)
results.append({
    "experiment":  "B4_signed_cos_sem",
    "outcome":     "is_additive",
    "predictors":  "signed_cos + sem_sim",
    "n_features":  2,
    "auc_loo":     exp8_auc_loo,
    "slope_abs_cos":         None,
    "slope_both_antisocial": None,
    "slope_signed_cos":      float(exp8_results.coef_[0, 0]),
    "slope_sem_sim":         float(exp8_results.coef_[0, 1]),
    **stats,
})

In [49]:
results_df = pd.DataFrame(results)
results_df

,experiment,outcome,predictors,n_features,auc_loo,auc_insample,ci_lo,ci_hi,n_boot,boot_aucs,...,loo_degen_folds,loo_n_folds,slope_abs_cos,slope_both_antisocial,slope_signed_cos,loo_slope_min,loo_slope_max,loo_slope_n_neg,loo_slope_n_folds,slope_sem_sim
0,M1_multi_signed_cos,regime,signed_cos,1,0.326639,0.648276,0.576537,0.846782,1889,"[0.6404794610151753, 0.5585754598662207, 0.751...",...,0,28.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,M2_multi_signed_sem,regime,signed_cos + sem_sim,2,0.409099,0.723427,0.653595,0.902520,1889,"[0.6753793343627102, 0.8043055555555555, 0.648...",...,0,28.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,M3_multi_abs_cos,regime,cosine_abs,1,0.079014,0.563165,0.522401,0.807130,1909,"[0.712989457146495, 0.6098922578184591, 0.6974...",...,0,28.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,M4_multi_abs_sem,regime,cosine_abs + sem_sim,2,0.295918,0.655770,0.608565,0.881622,1891,"[0.7806003631516402, 0.779023864980775, 0.7505...",...,0,28.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,B1_abs_cos,is_additive,cosine_abs,1,0.244898,0.619048,0.455612,0.829932,2000,"[0.7210884353741497, 0.7619047619047619, 0.598...",...,0,NaN,0.238507,NaN,NaN,0.073984,0.541459,0.0,28.0,NaN
5,B2_abs_cos_sem,is_additive,cosine_abs + sem_sim,2,0.653061,0.741497,0.598469,0.945578,2000,"[0.6802721088435374, 0.8299319727891157, 0.632...",...,0,NaN,0.930030,NaN,NaN,0.694687,1.380162,0.0,28.0,-1.300610
6,B3_signed_cos,is_additive,signed_cos,1,0.333333,0.619048,0.482993,0.863946,2000,"[0.5306122448979591, 0.782312925170068, 0.5102...",...,0,NaN,NaN,NaN,0.339155,0.173809,0.694546,0.0,28.0,NaN
7,B4_signed_cos_sem,is_additive,signed_cos + sem_sim,2,0.517007,0.727891,0.571429,0.925170,2000,"[0.6054421768707483, 0.8231292517006803, 0.734...",...,0,NaN,NaN,NaN,0.690960,0.440258,1.223362,0.0,28.0,-1.022784


In [50]:
print("Total pairs:", len(df))
print("\nRegime counts:")
print(df['regime'].value_counts())
print("\nBinary balance:")
print(df['is_additive'].value_counts())
print("\nMissing values:")
print(df[['cosine_abs', 'sem_sim', 'is_additive', 'cos', 'regime']].isna().sum())
print("\ncos distribution:")
print(df['cos'].describe())

Total pairs: 28

Regime counts:
regime
mixed          14
additive        7
suppressive     4
dominant        3
Name: count, dtype: int64

Binary balance:
is_additive
0    21
1     7
Name: count, dtype: int64

Missing values:
cosine_abs     0
sem_sim        0
is_additive    0
cos            0
regime         0
dtype: int64

cos distribution:
count    28.000000
mean      0.150936
std       0.252215
min      -0.522500
25%      -0.001975
50%       0.209250
75%       0.305325
max       0.694800
Name: cos, dtype: float64



In order to study the underlying geometry of the space where steering vectors live, we follow two different directions:
- How strongly do the behaviours come out together?
- Do the behaviours reinforce or suppress one another?